## Visão Geral do Microsoft Entra ID

Microsoft Entra ID (anteriormente Azure Active Directory) é o serviço de gerenciamento de identidade e acesso baseado em nuvem da Microsoft. Ele serve como provedor de identidade central para Microsoft 365, Azure e milhares de outras aplicações SaaS.

Recursos Principais:
* **Single Sign-On (SSO)** - Usuários se autenticam uma vez para acessar múltiplas aplicações.
* **Multi-Factor Authentication (MFA)** - Segurança aprimorada através de métodos adicionais de verificação.
* **Conditional Access** - Controle de acesso baseado em políticas com base em usuário, dispositivo, localização e risco.
* **Application Integration** - Suporta protocolos de autenticação modernos como OAuth 2.0, OpenID Connect e SAML.

## Objetivo de Aprendizado

Microsoft Entra ID pode ser usado como provedor de identidade no AgentCore Identity e usado para autenticar usuários e fazê-los autorizar o agente a acessar recursos protegidos em seu nome. Neste notebook, exploraremos o uso do Entra ID para autenticação de entrada.

- Autenticar usuários antes que possam invocar um agente.

## Fluxo Authorization Code

O fluxo authorization code do OAuth 2.0 é a abordagem recomendada para aplicações web autenticarem usuários com segurança e obterem tokens de acesso.

Este fluxo envolve:

1. Redirecionar usuários para o Entra ID para autenticação
2. Receber um código de autorização após login bem-sucedido
3. Trocar o código por tokens de acesso e refresh
4. Usar tokens para acessar recursos protegidos

Este padrão de integração permite que o AgentCore aproveite as robustas capacidades de gerenciamento de identidade do Entra ID enquanto mantém autenticação segura e baseada em padrões para suas aplicações.

## Objetivo de Aprendizado 1: Configurar Entra ID para uso com AgentCore Identity

### Passo 1: Configurar Tenant do Entra ID

Um tenant do Entra ID é uma instância dedicada do Microsoft Entra ID que representa sua organização. Pense nele como o diretório isolado da sua organização na nuvem da Microsoft.

Características Principais:

* **Identidade Única** - Cada tenant tem um domínio único (por exemplo, suaempresa.onmicrosoft.com)
* **Limite Isolado** - Usuários, grupos e aplicações em um tenant são separados de outros
* **Controle Administrativo** - Administradores do tenant gerenciam usuários, políticas de segurança e registros de aplicação
* **Suporte Multi-Domínio** - Pode incluir domínios personalizados além do domínio padrão .onmicrosoft.com

Na Prática:

Quando você registra uma aplicação com o Entra ID para integração OAuth 2.0, você está registrando-a dentro de um tenant específico. Usuários desse tenant podem então se autenticar em sua aplicação usando suas credenciais organizacionais.

Para integração com AgentCore, você precisará de:

* **Tenant ID** - Identificador único para a instância do Entra ID
* **Application Registration** - Sua aplicação registrada dentro do tenant
* **Permissões Apropriadas** - Direitos de acesso configurados para sua aplicação

Este modelo baseado em tenant garante que autenticação e autorização permaneçam dentro do limite de segurança da sua organização.

Passos para criar um tenant podem ser encontrados em https://learn.microsoft.com/en-us/entra/fundamentals/create-new-tenant.

Nota:
1. Microsoft Entra ID NÃO é um serviço AWS. Consulte a documentação do Microsoft Entra ID para informações relacionadas a custos.
2. Capturas de tela usadas nos seguintes passos podem mudar. Encorajamos você a consultar a documentação do Microsoft Entra ID para orientações mais recentes sobre configuração de uma aplicação Entra ID.

### Passo 2: Configurar Aplicação

1. Vá para https://portal.azure.com e pesquise por "Entra ID" na barra de pesquisa no topo da tela
<img src="images/entraid.jpg" width="75%">

2. Vá para `Manage` &rarr; `App Registrations`
<img src="images/app.registration.png" width="75%">

3. Clique em `New Registration` e preencha os detalhes. Certifique-se de selecionar a opção multi tenant
<img src="images/app.registration.form.png" width="75%">

4. Crie um client secret. Copie o clientId e client secret para uso no AgentCore Identity.
<img src="images/gather.client.info.png" width="75%">

5. Crie Scopes para OAuth. Vá para Expose an API &rarr; `Add Scope`. Copie e salve o scope completo.
<img src="images/expose.api.png" width="75%">

## Objetivo de Aprendizado 2 - Configurar um agente simples com Entra ID para autenticação de entrada

#### Pré-requisitos

* Python 3.10+
* Credenciais AWS
* Amazon Bedrock AgentCore SDK
* Strands Agents
* Definir região AWS como "us-west-2" ou qualquer região que suporte Bedrock AgentCore. Consulte https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agentcore-regions.html para regiões suportadas.
* Docker, Finch ou Podman instalado

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet # in quite mode to reduce output to the console/notebook

In [ ]:
import os
import uuid
import boto3
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime

boto_session = Session()
sts = boto3.client('sts')
account_id = sts.get_caller_identity().get("Account")
region = boto_session.region_name or "us-west-2"

print(f"AWS Region: {region}")
print(f"AWS Account: {account_id}")

#### Definindo variáveis de ambiente para algumas informações chave que precisaremos ao longo deste notebook.
Observe que o audience será o mesmo que "Application ID URI" do Passo 2.5 acima.

recuperar:
- Tenant ID de "App registration" --> "All Applications" --> Selecione o cliente que você acabou de criar --> "Overview" --> "Directory (tenant) ID"
- Client ID de "App registration" --> "All Applications" --> Selecione o cliente que você acabou de criar --> "Overview" --> "Application (client) ID"
- Secret salvo do passo anterior
- Scope será "Application ID URI" de "App registration" --> "All Applications" --> Selecione o cliente que você acabou de criar --> "Expose a API". Será sufixado com "/.default"
- Audience será "Application ID URI" de "App registration" --> "All Applications" --> Selecione o cliente que você acabou de criar --> "Expose a API".

In [ ]:
import os

# REPLACE WITH YOUR client_id
os.environ["client_id"] = "REPLACE_ME"

# REPLACE WITH YOUR secret
os.environ["secret"] = "REPLACE_ME"

# REPLACE WITH YOUR scopes
os.environ["scopes"] = "REPLACE_ME"

# REPLACE WITH YOUR tenant_id
os.environ["tenant_id"] = "REPLACE_ME"

# REPLACE WITH YOUR audience
os.environ["audience"] = "REPLACE_ME"

#### Código do Agente
Mantendo o agente simples já que o objetivo de aprendizado chave para este notebook é aprender autenticação de entrada usando EntraID

In [ ]:
%%writefile strands_wo_memory.py
import asyncio

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent

app = BedrockAgentCoreApp()
agent = Agent()

class StreamingQueue:
    def __init__(self):
        self.finished = False
        self.queue = asyncio.Queue()
        
    async def put(self, item):
        await self.queue.put(item)

    async def finish(self):
        self.finished = True
        await self.queue.put(None)

    async def stream(self):
        while True:
            item = await self.queue.get()
            if item is None and self.finished:
                break
            yield item

queue = StreamingQueue()

async def agent_task(user_message: str):
    try:
        await queue.put("Agent execution begins....")
        
        response = agent(user_message)
        # ... process agent response here ...
        await queue.put(response.message)
    except Exception as e:
        await queue.put(f"Failed with error: {repr(e)}")
    finally:
        await queue.put("Agent excecution finished")
        await queue.finish()

@app.entrypoint
async def strands_agent_bedrock(payload, context):
    print("Context object is ....", context)
    prompt = payload.get("prompt", "hello")
    
    task = asyncio.create_task(agent_task(prompt))
    
    async def stream_with_task():
        async for item in queue.stream():
            yield item
        await task
    
    return stream_with_task()

if __name__ == "__main__":
    app.run()


#### Configure seu runtime com `authorizer_configuration` para impor autenticação de entrada.
Você usará um `customJWTAuthorizer` para autenticação de entrada usando EntraID. Note como o discovery_url é construído usando Tenant ID

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()

discovery_url = f"https://login.microsoftonline.com/{os.environ['tenant_id']}/.well-known/openid-configuration"

response = agentcore_runtime.configure(
    entrypoint="strands_wo_memory.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="strands_wo_memory_entra_inbound",
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": discovery_url,
            "allowedAudience": os.environ["audience"].split(" "),
            # More JWT authorization can be added, refer to:
            # https://docs.aws.amazon.com/bedrock-agentcore-control/latest/APIReference/API_CustomJWTAuthorizerConfiguration.html
        }
    },
)

print(f"Runtime Agent: {response}")

## Objetivo de Aprendizado 3 - Implantar agente e invocar usando token bearer recebido anteriormente

#### Implantar Runtime Agent
Docker local precisa estar rodando já que local_build está habilitado. Como alternativa, você pode ter `local_build=False` para usar CloudBuild.

In [ ]:
strands_wo_memory_launch_response = agentcore_runtime.launch(
    local_build=False,
    auto_update_on_conflict=True,
)

#### Use o SDK MSAL para obter um código de autorização do Entra ID

In [ ]:
import msal

REDIRECT_URI = f"https://bedrock-agentcore.{region}.amazonaws.com/identities/oauth2/callback"
AUTHORITY = f"https://login.microsoftonline.com/{os.environ['tenant_id']}"

app = msal.ConfidentialClientApplication(
    os.environ['client_id'],
    authority=AUTHORITY,
    client_credential=os.environ["secret"], 
)

result = app.acquire_token_for_client(scopes=[os.environ["scopes"]]) # Note that scope expects a list and not string
bearer_token_entra = result['access_token']

In [ ]:
import urllib.parse
import requests
import json
import uuid

if not strands_wo_memory_launch_response.agent_arn:
    raise Exception(
        "Missing Runtime Agent ARN. Verify that the Runtime Agent was created successfully in the previous step."
    )

escaped_agent_arn = urllib.parse.quote(
    strands_wo_memory_launch_response.agent_arn, safe=""
)
url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{escaped_agent_arn}/invocations?qualifier=DEFAULT"

session_id = str(uuid.uuid1())
headers = {
    "Authorization": f"Bearer {bearer_token_entra}",
    "Content-Type": "application/json",
    "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    "X-Amzn-Trace-Id": f"entra_id_inbound_sample_{session_id}",
}

http_response = requests.post(
    url,
    data=json.dumps(
        {"prompt": "Hello! I am John Doe. I like brick oven pizza!", "user_id": "user1"}
    ),
    headers=headers,
)
http_response.raise_for_status()


print(f"Agent Response: {http_response.text}")

#### Interações anteriores nesta sessão estão disponíveis através de agents.messages. Memória do AgentCore não é usada. O agente não se lembrará de interações anteriores se um novo session ID for usado.

In [ ]:
http_response = requests.post(
    url, data=json.dumps({"prompt": "Who am I?", "user_id": "user1"}), headers=headers
)
http_response.raise_for_status()

print(f"Agent Response: {http_response.text}")

#### Como alternativa, você pode usar o objeto AgentCore Runtime para invocar o agente. Passe o bearer token e o mesmo session ID para continuar com a sessão anterior.

In [ ]:
invoke_response = agentcore_runtime.invoke(
    {"prompt": "Who am I?", "user_id": "user1"},
    bearer_token=bearer_token_entra,
    session_id=session_id,
)

print(f"Agent Invoke Response: {invoke_response}")

## Conclusão e Limpeza

Neste notebook aprendemos como:

- Configurar API e Aplicação do Entra ID para fornecer fluxo OAuth 2.0 Authorization Code
- Criar um AgentCore Runtime e Implantar um agente com autenticação de entrada usando Entra ID
- Obter um token para acessar o Agente protegido

#### Recurso(s) criado(s)

In [ ]:
print(f"Runtime Agent Arn: {strands_wo_memory_launch_response.agent_id}")

#### Deletar AgentCore Runtime Agent

In [ ]:
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)

agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=strands_wo_memory_launch_response.agent_id
)